# PISA 2025 Insight Report: Mongolia and Rwanda

This analysis compares PISA 2025 mathematics performance in Mongolia and Rwanda, with a pooled OECD comparison group as a broader benchmark. Mathematics is the primary outcome, while reading and science provide context. The analysis examines weighted achievement, socioeconomic status (ESCS), gender, grade repetition, school location, educational material shortages, and staff shortages. Results are descriptive rather than causal. PISA student weights and all 10 plausible values are used for achievement summaries, while correlations and subgroup comparisons describe associations with mathematics performance.

# 1. Package Imports

In [ ]:
# importing/installing the required packages
!pip install pyreadstat

import pandas as pd
import gdown # used to download the PISA files directly from Google drive
import pyreadstat
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Downloading PISA Files

In [ ]:
# assigning file names in this Colab session
student_file = "CY09_MS_STU_PUF.sav"
school_file = "CY09_MS_SCH_PUF.sav"

# download the files from the shared Google Drive
gdown.download(
    id="163VOjSHyLiuVDj4pZRX9g5sSeMBn3FXO",
    output=student_file
)

gdown.download(
    id="1VZyjoUiPhUBnxFBOJPqmRBDjAmGg0smZ",
    output=school_file
)

# 3. Data Loading and Column Selection


## 3A. Load School Data

In [ ]:
# loading the PISA school dataset
school_df = pd.read_spss(
   school_file,
   convert_categoricals=False
)

school_df = school_df[["CNT", "OECD", "CNTSCHID", "SC001Q01TA", "EDUSHORT", "STAFFSHORT"]]

# school integrity checks
print(school_df.shape)
display(school_df.head())

## 3B. Select and Load Student Data

In [ ]:
# defining the student columns so we don't load the entire student database at once
student_cols = [
    # IDs / country
    "CNT",
    "OECD",
    "CNTSCHID",
    "CNTSTUID",

    # Student characteristics
    "ESCS",
    "MALE",
    "REPEAT",

    # Sampling weight
    "W_FSTUWT",

    # Mathematics plausible values
    "PV1MATH", "PV2MATH", "PV3MATH", "PV4MATH", "PV5MATH",
    "PV6MATH", "PV7MATH", "PV8MATH", "PV9MATH", "PV10MATH",

    # Reading plausible values
    "PV1READ", "PV2READ", "PV3READ", "PV4READ", "PV5READ",
    "PV6READ", "PV7READ", "PV8READ", "PV9READ", "PV10READ",

    # Science plausible values
    "PV1SCIE", "PV2SCIE", "PV3SCIE", "PV4SCIE", "PV5SCIE",
    "PV6SCIE", "PV7SCIE", "PV8SCIE", "PV9SCIE", "PV10SCIE"
]

# loading the PISA student dataset
student_df = pd.read_spss(
    student_file,
    usecols=student_cols,
    convert_categoricals=False
)

# student df integrity checks
print(student_df.shape)
display(student_df.head())

# 4. Column Standardization


## 4A. Standardize School Column Names

In [ ]:
#rename school columns
school_df = school_df.rename(columns={
    "CNT": "country",
    "OECD": "is_oecd",
    'CNTSCHID': 'school_id',
    'SC001Q01TA': 'school_location',
    'EDUSHORT': 'edu_material_shortage',
    'STAFFSHORT': 'staff_shortage'
})

# school df integrity check
school_df.head()

## 4B. Standardize Student Column Names

In [ ]:
# rename student columns

rename_dict = {
    'CNT': 'country',
    'OECD' : 'is_oecd',
    'CNTSCHID': 'school_id',
    'CNTSTUID': 'student_id',
    'MALE': 'is_male',
    'ESCS': 'socioeconomic_index',
    'REPEAT': 'grade_repetition',
    'W_FSTUWT': 'student_weight'
}

for i in range(1, 11):
    rename_dict[f'PV{i}SCIE'] = f'science{i}'
    rename_dict[f'PV{i}MATH'] = f'math{i}'
    rename_dict[f'PV{i}READ'] = f'read{i}'

student_df = student_df.rename(columns=rename_dict)

# student df integrity check
student_df.head()

# 5. Data Cleaning & Missingness Analysis


## 5A. Define Missing-Value Codes

In [ ]:
# common PISA missing-value codes
missing_95_99 = [95, 96, 97, 98, 99]
missing_repeat = [5, 6, 7, 8, 9]
missing_plausible_value = [9997]
missing_school_id = ['9999997']

# missing-value code translation:
# 95 = valid skip
# 96 = system missing
# 97 = not administered / not applicable
# 98 = invalid
# 99 = no response

# for repeat:
# 5 = valid skip
# 6 = system missing
# 7 = not administered / not applicable
# 8 = invalid
# 9 = no response

# for plausible values:
# 9997 = not administered

## 5B. Map Missing Codes to Variables

In [ ]:
# mapping missing codes for school variables
school_missing_values = {
    "school_id": missing_school_id,
    "school_location": missing_95_99,
    "edu_material_shortage": missing_95_99,
    "staff_shortage": missing_95_99
}


# mapping missing codes for student variables
student_missing_values = {
    "country": [],
    "is_oecd": [],
    "school_id": missing_school_id,
    "student_id": [],
    "socioeconomic_index": missing_95_99,
    "is_male": missing_95_99,
    "grade_repetition": missing_repeat,
    "student_weight": []
}

for i in range(1, 11):
    student_missing_values[f"math{i}"] = missing_plausible_value
    student_missing_values[f"read{i}"] = missing_plausible_value
    student_missing_values[f"science{i}"] = missing_plausible_value

## 5C. Create Analysis Groups

In [ ]:
# creating analysis groups for school data
school_df["analysis_group"] = np.select(
    [
        school_df["country"] == "MNG",
        school_df["country"] == "RWA",
        school_df["is_oecd"] == 1
    ],
    [
        "Mongolia",
        "Rwanda",
        "OECD"
    ],
    default="Other"
)


# creating analysis groups for student data
student_df["analysis_group"] = np.select(
    [
        student_df["country"] == "MNG",
        student_df["country"] == "RWA",
        student_df["is_oecd"] == 1
    ],
    [
        "Mongolia",
        "Rwanda",
        "OECD"
    ],
    default="Other"
)

## 5D. Clean School Data and Check Missingness

In [ ]:
# assigning NA values to school_df missing codes
school_clean = school_df.copy()

for column, missing_codes in school_missing_values.items():
    school_clean[column] = school_clean[column].replace(
        missing_codes,
        np.nan
    )

# defining school columns for the NA analysis
school_missing_cols = [
    "school_location",
    "edu_material_shortage",
    "staff_shortage"
]

# mean school df missingness
school_missing = (
    school_clean[
        school_clean["analysis_group"].isin(
            ["Mongolia", "Rwanda", "OECD"]
        )
    ]
    .groupby("analysis_group")[school_missing_cols]
    .agg(lambda x: x.isna().mean() * 100)
    .round(2)
)

# check school ids
school_id_missing = (
    school_clean
    .groupby("analysis_group")[["school_id"]]
    .agg(lambda x: x.isna().mean() * 100)
    .round(2)
)

print("=== Missing data for School Dataframe ===")
display(school_missing)
print("=== Missing ids for School Dataframe ===")
display(school_id_missing.loc[["Mongolia", "Rwanda", "OECD"]])

# ensuring each school id appears only once
assert school_clean["school_id"].dropna().is_unique

## 5E. School Missingness Visualization

In [ ]:
# keeping only missing school variables with some missing data
missing_school_plot = school_missing.loc[
    :, school_missing.max() > 0
]

# creating school missingness plot
plt.figure(figsize=(7, 3))
plt.imshow(missing_school_plot, aspect="auto", cmap="RdYlBu_r", vmin=0, vmax=15)
plt.xticks(
    range(len(missing_school_plot.columns)),
    missing_school_plot.columns,
    rotation=45,
    ha="right"
)
plt.yticks(
    range(len(missing_school_plot.index)),
    missing_school_plot.index
)

# customizing school missingness plot
plt.colorbar(label="Percent Missing")
plt.title("School Data Missingness by Analysis Group")
plt.tight_layout()
plt.show()

## 5F. Clean Student Data and Check Missingness

In [ ]:
# assigning NA values to student_df missing codes

student_clean = student_df.copy()
for column, missing_codes in student_missing_values.items():
    student_clean[column] = (
        student_clean[column]
        .replace(missing_codes, np.nan)
    )

# defining student columns for the NA analysis
math_cols = [f"math{i}" for i in range(1, 11)]
read_cols = [f"read{i}" for i in range(1, 11)]
science_cols = [f"science{i}" for i in range(1, 11)]

student_missing_cols = (
    ["socioeconomic_index", "is_male", "grade_repetition"]
    + math_cols
    + read_cols
    + science_cols
)

# mean student df missingness
student_missing = (
    student_clean[
        student_clean["analysis_group"].isin(
            ["Mongolia", "Rwanda", "OECD"]
        )
    ]
    .groupby("analysis_group")[student_missing_cols]
    .agg(lambda x: x.isna().mean() * 100)
    .round(2)
)

# check missing identifiers
student_id_cols = [
    "country",
    "is_oecd",
    "school_id",
    "student_id",
    "student_weight"
]

# identifiers mean missingness
student_id_missing = (
    student_clean
    .groupby("analysis_group")[student_id_cols]
    .agg(lambda x: x.isna().mean() * 100)
    .round(2)
)

# printing tables for verification
print("=== Missing data for Student Dataframe ===")
display(student_missing)
print("=== Missing ids for Student Dataframe ===")
display(student_id_missing.loc[["Mongolia", "Rwanda", "OECD"]])

## 5G. Student Missingness Visualization

In [ ]:
# keeping only missing student variables with some missing data
missing_student_plot = student_missing.loc[
    :, student_missing.max() > 0
]

# creating student missingness plot
plt.figure(figsize=(10, 4))
plt.imshow(missing_student_plot, aspect="auto", cmap="RdYlBu_r", vmin=0, vmax=15)
plt.xticks(
    range(len(missing_student_plot.columns)),
    missing_student_plot.columns,
    rotation=45,
    ha="right"
)
plt.yticks(
    range(len(missing_student_plot.index)),
    missing_student_plot.index
)

# customizing student missingness plot
plt.colorbar(label="Percent Missing (%)")
plt.title("Student Data Missingness by Analysis Group")
plt.xlabel("Variable")
plt.ylabel("Analysis Group")

# displaying student missingness plot
plt.tight_layout()
plt.show()

### Analysis Note: Data Missingness and Integrity

School identifiers are complete for Mongolia and Rwanda. Among OECD students, 3.19% lack a valid school identifier, which reduces coverage for analyses that attach school characteristics. Mongolia and Rwanda have at least 90% observed values for school location, educational material shortage, and staff shortage. OECD school-level missingness is higher: 12.04% for location, 13.67% for material shortage, and 13.59% for staff shortage.

Student identifiers, weights, gender, and all 30 plausible values are complete. Grade-repetition missingness is 0.50% in Mongolia, 5.06% in Rwanda, and 9.52% in the OECD comparison group. ESCS missingness is 3.43%, 6.67%, and 9.63%, respectively. Because students with missing ESCS later show lower mathematics scores than students with observed ESCS, socioeconomic comparisons may underrepresent some lower-performing students. Overall achievement estimates are unaffected by missing plausible values or weights, while school-condition and ESCS analyses use reduced samples.

# 6. Weighted Plausible Value Mean Estimation

For each subject and analysis group, the student-weighted mean is calculated separately for all 10 plausible values and then averaged across the 10 estimates.

## 6A. Weighted Plausible Value Function

In [ ]:
# calculating weighted mean achievement across the 10 plausible values
def weighted_subject_mean(data, analysis_group, subject="math"):
    """Calculates the weighted mean score across a subject's 10 plausible values."""

    group_rows = data[data["analysis_group"] == analysis_group]
    score_columns = [f"{subject}{i}" for i in range(1, 11)]
    weighted_scores = []

    # weighted mean for each plausible value
    for score_column in score_columns:
        valid_rows = group_rows.dropna(
            subset=[score_column, "student_weight"]
        )

        weighted_score = np.average(
            valid_rows[score_column],
            weights=valid_rows["student_weight"]
        )

        weighted_scores.append(weighted_score)

    return np.mean(weighted_scores)

## 6B. Overall PISA Performance Summary

In [ ]:
# calculating weighted means across subjects and groups
subjects = ["math", "read", "science"]
groups = ["Rwanda", "Mongolia", "OECD"]
performance_results = {}

for group in groups:
    performance_results[group] = {}

    for subject in subjects:
        performance_results[group][subject] = weighted_subject_mean(
            student_clean,
            group,
            subject
        )


# creating weighted performance summary
performance_summary = pd.DataFrame(performance_results).T[
    ["math", "read", "science"]
]

performance_summary.columns = [
    "Mathematics",
    "Reading",
    "Science"
]

print("=== PISA 2025 Weighted Performance Summary ===")
display(performance_summary.round(2))

**Analysis Note: PISA Weighted Performance**

Mongolia substantially outperforms Rwanda across all three domains. The mathematics gap is **106.99 points** (418.64 vs. 311.65), the science gap is **106.74 points** (424.03 vs. 317.29), and the reading gap is **73.31 points** (374.25 vs. 300.94). Both countries score highest in science and lowest in reading.

Mongolia also remains below the OECD comparison group by **39.73 points in mathematics**, **60.43 in science**, and **92.96 in reading**. Rwanda's gaps with the OECD comparison group are larger: **146.72 in mathematics**, **166.27 in reading**, and **167.17 in science**. Mathematics is the primary outcome for the remaining analysis.

# 7. Comparison of Student and School Characteristics

## 7A. Weighted Summary Functions

In [ ]:
# calculating weighted population mean
def weighted_population_mean(
    data,
    value_column,
    weight_column="student_weight"
):
    """Calculates a population-weighted mean."""

    valid_rows = data.dropna(
        subset=[value_column, weight_column]
    )

    return np.average(
        valid_rows[value_column],
        weights=valid_rows[weight_column]
    )


# calculating weighted percentage for one category
def weighted_percentage(
    data,
    category_column,
    category_value,
    weight_column="student_weight"
):
    """Calculates the weighted percentage in one category."""

    valid_rows = data.dropna(
        subset=[category_column, weight_column]
    )

    matching_weight = valid_rows[
        valid_rows[category_column] == category_value
    ][weight_column].sum()

    total_weight = valid_rows[weight_column].sum()

    return (matching_weight / total_weight) * 100

## 7B. Student and School Characteristics Summary

In [ ]:
# calculating student and school characteristics by group
groups = ["Mongolia", "Rwanda", "OECD"]
summary_rows = []

for group in groups:
    school_rows = school_clean[
        school_clean["analysis_group"] == group
    ]
    student_rows = student_clean[
        student_clean["analysis_group"] == group
    ]
    # attaching school characteristics to students
    student_school_rows = student_rows[
        ["school_id", "student_weight"]
    ].merge(
        school_rows[
            [
                "school_id",
                "school_location",
                "edu_material_shortage",
                "staff_shortage"
            ]
        ],
        on="school_id",
        how="inner",
        validate="many_to_one"
    )

    # storing summary statistics for each group
    summary_rows.append({
        "Analysis Group": group,
        "Male Students (%)": weighted_percentage(
            student_rows,
            "is_male",
            1.0
        ),
        "Grade Repetition Rate (%)": weighted_percentage(
            student_rows,
            "grade_repetition",
            1.0
        ),
        "Mean ESCS (Socioeconomic)": weighted_population_mean(
            student_rows,
            "socioeconomic_index"
        ),
        "City or Larger School Location (%)": (
            weighted_percentage(
                student_school_rows,
                "school_location",
                4.0
            )
            +
            weighted_percentage(
                student_school_rows,
                "school_location",
                5.0
            )
            +
            weighted_percentage(
                student_school_rows,
                "school_location",
                6.0
            )
        ),
        "Educational Material Shortage (Mean)": weighted_population_mean(
            student_school_rows,
            "edu_material_shortage"
        ),
        "Staff Shortage Index (Mean)": weighted_population_mean(
            student_school_rows,
            "staff_shortage"
        )
    })


# creating student and school characteristics summary
covariate_summary = (
    pd.DataFrame(summary_rows)
    .set_index("Analysis Group")
)

display(covariate_summary.round(2))

### Analysis Note: Student and School Characteristics

Rwanda differs sharply from Mongolia in several student and school characteristics. Grade repetition is **55.70%** in Rwanda versus **5.65%** in Mongolia and **11.18%** in the OECD comparison group. Mean ESCS is also much lower in Rwanda (**−2.15**) than Mongolia (**−0.78**) or the OECD comparison group (**−0.19**).

Only **14.41%** of Rwanda's represented students attend schools classified as city or larger, compared with **47.06%** in Mongolia and **48.94%** in the OECD comparison group. Rwanda also reports higher educational material shortages (**0.67**) than Mongolia (**0.04**) and the OECD comparison group (**−0.07**). Staff-shortage means are **0.26** in Rwanda, **0.06** in Mongolia, and **0.28** in the OECD comparison group. These descriptive differences motivate examining mathematics performance across student and school conditions.


# 8. Association with Mathematics Performance

## 8A. Merge Student and School Data

In [ ]:
# merging school characteristics onto student data
student_school_data = student_clean.merge(
    school_clean[
        [
            "school_id",
            "school_location",
            "edu_material_shortage",
            "staff_shortage"
        ]
    ],
    on="school_id",
    how="left",
    validate="many_to_one"
)

# checking that the merge did not create extra student rows
assert len(student_school_data) == len(student_clean)

## 8B. Weighted Mathematics Performance by Category

In [ ]:
# calculating weighted math performance by category
def weighted_math_by_category(
    data,
    category_column,
    analysis_group
):
    """Calculates weighted mathematics performance for each category of a variable."""

    group_rows = data[
        data["analysis_group"] == analysis_group
    ]

    math_columns = [
        f"math{i}" for i in range(1, 11)
    ]

    category_scores = {}

    for category, category_rows in group_rows.groupby(
        category_column,
        observed=True
    ):
        weighted_math_scores = []

        for math_column in math_columns:
            valid_rows = category_rows.dropna(
                subset=[math_column, "student_weight"]
            )

            if len(valid_rows) > 0:
                weighted_score = np.average(
                    valid_rows[math_column],
                    weights=valid_rows["student_weight"]
                )

                weighted_math_scores.append(weighted_score)

        if weighted_math_scores:
            category_scores[category] = np.mean(
                weighted_math_scores
            )

    return category_scores

## 8C. Gender and Grade Repetition

In [ ]:
# calculating math performance by gender and grade repetition
groups = ["Mongolia", "Rwanda", "OECD"]
subgroup_rows = []

for group in groups:
    gender_scores = weighted_math_by_category(
        student_school_data,
        "is_male",
        group
    )

    repetition_scores = weighted_math_by_category(
        student_school_data,
        "grade_repetition",
        group
    )

    subgroup_rows.append({
        "Analysis Group": group,
        "Female Math": gender_scores.get(0.0, np.nan),
        "Male Math": gender_scores.get(1.0, np.nan),
        "Gender Gap (Male - Female)":
            gender_scores.get(1.0, np.nan)
            - gender_scores.get(0.0, np.nan),
        "No Repetition": repetition_scores.get(0.0, np.nan),
        "Repeated Grade": repetition_scores.get(1.0, np.nan),
        "Repetition Gap (Repeated - No Repetition)":
            repetition_scores.get(1.0, np.nan)
            - repetition_scores.get(0.0, np.nan)
    })

subgroup_math_summary = (
    pd.DataFrame(subgroup_rows)
    .set_index("Analysis Group")
)

print("=== Mathematics Performance by Gender and Grade Repetition ===")
display(subgroup_math_summary.round(2))

## 8D. Mathematics Performance Across ESCS Ranges

In [ ]:
# creating common ESCS ranges
escs_bins = [
    -np.inf,
    -2.0,
    -1.0,
    0.0,
    1.0,
    np.inf
]

escs_labels = [
    "Below -2.0",
    "-2.0 to -1.0",
    "-1.0 to 0.0",
    "0.0 to 1.0",
    "Above 1.0"
]

student_school_data["escs_range"] = pd.cut(
    student_school_data["socioeconomic_index"],
    bins=escs_bins,
    labels=escs_labels
)


# calculating math performance across common ESCS ranges
math_by_escs = pd.DataFrame({
    group: weighted_math_by_category(
        student_school_data,
        "escs_range",
        group
    )
    for group in groups
})

print("=== Mathematics Performance Across Common ESCS Ranges ===")
display(math_by_escs.round(2))


# counting students in each ESCS range by group
escs_range_counts = (
    student_school_data
    .groupby(
        ["analysis_group", "escs_range"],
        observed=True
    )
    .size()
    .unstack()
    .reindex(groups)
)

print("=== Student Counts Across Common ESCS Ranges ===")
display(escs_range_counts)

## 8E. ESCS Missingness Check

In [ ]:
# identifying students with missing and observed ESCS values
student_school_data["escs_status"] = np.where(
    student_school_data["socioeconomic_index"].isna(),
    "ESCS Missing",
    "ESCS Observed"
)

# comparing math performance by ESCS missingness
math_by_escs_status = pd.DataFrame({
    group: weighted_math_by_category(
        student_school_data,
        "escs_status",
        group
    )
    for group in groups
})

print("=== Mathematics Performance by ESCS Missingness ===")
display(math_by_escs_status.round(2))

## 8F. Educational Material Shortages

In [ ]:
# creating common educational material shortage categories
material_shortage_bins = [
    -np.inf,
    -0.5,
    0.5,
    np.inf
]

material_shortage_labels = [
    "Low",
    "Moderate",
    "High"
]

student_school_data["material_shortage_category"] = pd.cut(
    student_school_data["edu_material_shortage"],
    bins=material_shortage_bins,
    labels=material_shortage_labels
)


# calculating math performance by educational material shortage
math_by_material_shortage = pd.DataFrame({
    group: weighted_math_by_category(
        student_school_data,
        "material_shortage_category",
        group
    )
    for group in groups
})

print("=== Mathematics Performance by Educational Material Shortage ===")
display(math_by_material_shortage.round(2))

## 8G. Mathematics Performance by School Location

In [ ]:
# defining PISA school location labels
school_location_labels = {
    1.0: "Village/Rural (<3,000)",
    2.0: "Small Town (3,000-15,000)",
    3.0: "Town (15,000-100,000)",
    4.0: "City (100,000-1M)",
    5.0: "Large City (1M-10M)",
    6.0: "Megacity (>10M)"
}

school_location_order = [
    1.0,
    2.0,
    3.0,
    4.0,
    5.0,
    6.0
]


# calculating math performance by school location
math_by_school_location = pd.DataFrame({
    group: weighted_math_by_category(
        student_school_data,
        "school_location",
        group
    )
    for group in groups
}).reindex(school_location_order)


# adding readable location labels
math_by_school_location.index = [
    school_location_labels[location]
    for location in school_location_order
]

print("=== Mathematics Performance by School Location ===")
display(math_by_school_location.round(2))

## 8H. Weighted Correlation Analysis

In [ ]:
# calculating weighted Pearson correlation
def weighted_correlation(x, y, weights):
    """Calculates a weighted Pearson correlation."""

    x_mean = np.average(x, weights=weights)
    y_mean = np.average(y, weights=weights)

    weighted_covariance = np.average(
        (x - x_mean) * (y - y_mean),
        weights=weights
    )

    x_variance = np.average(
        (x - x_mean) ** 2,
        weights=weights
    )

    y_variance = np.average(
        (y - y_mean) ** 2,
        weights=weights
    )

    return weighted_covariance / np.sqrt(
        x_variance * y_variance
    )


# calculating weighted math correlation across the 10 plausible values
def weighted_math_correlation(
    data,
    analysis_group,
    variable_column
):
    """Calculates the average weighted correlation across the 10 math plausible values."""

    group_rows = data[
        data["analysis_group"] == analysis_group
    ]

    math_columns = [
        f"math{i}" for i in range(1, 11)
    ]

    correlations = []

    for math_column in math_columns:
        valid_rows = group_rows.dropna(
            subset=[
                math_column,
                variable_column,
                "student_weight"
            ]
        )

        correlation = weighted_correlation(
            valid_rows[math_column],
            valid_rows[variable_column],
            valid_rows["student_weight"]
        )

        correlations.append(correlation)

    return np.mean(correlations)


# calculating math correlations with student and school characteristics
correlation_rows = []

for group in groups:
    correlation_rows.append({
        "Analysis Group": group,

        "ESCS": weighted_math_correlation(
            student_school_data,
            group,
            "socioeconomic_index"
        ),

        "Material Shortage": weighted_math_correlation(
            student_school_data,
            group,
            "edu_material_shortage"
        ),

        "Staff Shortage": weighted_math_correlation(
            student_school_data,
            group,
            "staff_shortage"
        )
    })


# creating correlation summary
correlation_summary = (
    pd.DataFrame(correlation_rows)
    .set_index("Analysis Group")
)

print("=== Weighted Correlations with Mathematics Performance ===")
display(correlation_summary.round(3))


## 8I. ESCS and Mathematics Visualization

In [ ]:
# creating a student-level average math score for visualization only
# formal statistics still use the 10 plausible values separately
math_columns = [f"math{i}" for i in range(1, 11)]

student_school_data["math_average"] = student_school_data[
    math_columns
].mean(axis=1)


# using a colorblind-safe plotting style
plt.style.use("tableau-colorblind10")

group_colors = {
    "Mongolia": "#006BA4",
    "Rwanda": "#FF800E",
    "OECD": "#3EBCD2"
}


# plotting ESCS and mathematics separately for each analysis group
for group in groups:
    group_rows = student_school_data[
        student_school_data["analysis_group"] == group
    ].dropna(
        subset=["socioeconomic_index", "math_average"]
    )

    plt.figure(figsize=(8, 5))

    plt.scatter(
        group_rows["socioeconomic_index"],
        group_rows["math_average"],
        alpha=0.20,
        s=12,
        color=group_colors[group]
    )

    plt.xlabel("Socioeconomic Index (ESCS)")
    plt.ylabel("Average Mathematics Score")
    plt.title(f"Mathematics Performance and ESCS: {group}")

    # keeping the axes consistent across groups
    plt.xlim(-7, 5)
    plt.ylim(100, 850)

    plt.show()

### Analysis Note: Associations with Mathematics Performance

Gender differences are relatively small: Mongolia shows an **8.37-point female advantage**, while males score **22.97 points higher** in Rwanda and **18.70 points higher** in the OECD comparison group.

Grade repetition shows a larger association. Students who repeated a grade score **91.17 points lower** than non-repeaters in Mongolia, **45.87 points lower** in Rwanda, and **85.19 points lower** in the OECD comparison group. These differences are descriptive rather than causal.

Mathematics generally rises with socioeconomic status. Mongolia increases from **373.89** below ESCS −2.0 to **484.69** above 1.0, while the OECD comparison group rises from **381.73** to **516.69**. Rwanda rises from **308.22** below −2.0 to **356.10** between 0 and 1 before falling to **348.51** above 1.0; only **36 Rwandan students** are in that highest category. Weighted ESCS correlations are **0.376** in Mongolia, **0.143** in Rwanda, and **0.395** in the OECD comparison group. Students with missing ESCS also score lower in all three groups, so ESCS-based comparisons may underrepresent some lower-performing students.

Educational material shortages are negatively associated with mathematics. Scores fall from low- to high-shortage categories by **22.66 points in Mongolia**, **30.21 in Rwanda**, and **23.19 in the OECD comparison group**. Weighted correlations are **−0.124**, **−0.161**, and **−0.103**, respectively. Staff-shortage correlations are weaker at **−0.017**, **−0.098**, and **−0.049**.

School location also varies with performance. Rwanda rises from **304.00** in village/rural schools to **358.35** in large cities, although the pattern is not monotonic. Mongolia rises from **376.00** in village/rural schools to **438.09** in large cities, while the OECD comparison group ranges from **410.15** in village/rural schools to **480.11** in megacities. These associations do not establish causality.


### Policy Implications

For Rwanda, the results point to two priorities for further policy evaluation: educational material constraints and the mechanisms behind the unusually high rate of grade repetition, particularly among socioeconomically disadvantaged students. Because this analysis is observational, these patterns should guide further investigation and program evaluation rather than be interpreted as evidence that changing any single factor would cause higher mathematics performance.

# 9. Outcome Visualizations

## 9A. Overall PISA Performance

In [ ]:
# plotting overall weighted achievement across PISA domains
subjects = ["Mathematics", "Reading", "Science"]
x = np.arange(len(subjects))
width = 0.25

group_colors = {
    "Rwanda": "#FF800E",
    "Mongolia": "#006BA4",
    "OECD": "#3EBCD2"
}


# Figure 1A: bar chart with zero baseline
plt.figure(figsize=(9, 5))

bars_rwanda = plt.bar(
    x - width,
    performance_summary.loc["Rwanda", subjects],
    width,
    label="Rwanda",
    color=group_colors["Rwanda"]
)

bars_mongolia = plt.bar(
    x,
    performance_summary.loc["Mongolia", subjects],
    width,
    label="Mongolia",
    color=group_colors["Mongolia"]
)

bars_oecd = plt.bar(
    x + width,
    performance_summary.loc["OECD", subjects],
    width,
    label="OECD Comparison Group",
    color=group_colors["OECD"]
)

# adding value labels above bars
for bars in [bars_rwanda, bars_mongolia, bars_oecd]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height + 5,
            f"{height:.0f}",
            ha="center",
            fontsize=9
        )

plt.ylabel("Weighted Mean Score")
plt.title("Figure 1A: Overall Achievement Across PISA Domains")
plt.xticks(x, subjects)
plt.ylim(0, 550)
plt.legend()
plt.grid(axis="y", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

## 9B. Student Population Characteristics

In [ ]:
# Figure 2A and 2B: student population characteristics
plt.style.use("tableau-colorblind10")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))


# Figure 2A: weighted ESCS distributions
for group in ["Rwanda", "Mongolia", "OECD"]:
    group_rows = student_clean[
        student_clean["analysis_group"] == group
    ]

    display_label = (
        "OECD Comparison Group"
        if group == "OECD"
        else group
    )

    sns.kdeplot(
        data=group_rows,
        x="socioeconomic_index",
        weights="student_weight",
        ax=axes[0],
        label=display_label,
        fill=True,
        alpha=0.20
    )

axes[0].set_title(
    "Figure 2A: Socioeconomic Status Distribution",
    fontweight="bold"
)
axes[0].set_xlabel("Socioeconomic Index (ESCS)")
axes[0].set_ylabel("Density")
axes[0].legend()
axes[0].grid(linestyle=":", alpha=0.5)


# Figure 2B: grade repetition rates
repetition_rates = [
    covariate_summary.loc[
        "Rwanda",
        "Grade Repetition Rate (%)"
    ],
    covariate_summary.loc[
        "Mongolia",
        "Grade Repetition Rate (%)"
    ],
    covariate_summary.loc[
        "OECD",
        "Grade Repetition Rate (%)"
    ]
]

group_labels = [
    "Rwanda",
    "Mongolia",
    "OECD Comparison Group"
]

bars = axes[1].bar(
    group_labels,
    repetition_rates,
    width=0.5
)

axes[1].set_title(
    "Figure 2B: Grade Repetition Rates",
    fontweight="bold"
)
axes[1].set_ylabel("Percent of Students (%)")
axes[1].set_ylim(0, 65)
axes[1].grid(
    axis="y",
    linestyle=":",
    alpha=0.5
)

# adding percentage labels
for bar, rate in zip(bars, repetition_rates):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        rate + 1,
        f"{rate:.1f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

## 9C. Mathematics Across ESCS Ranges

In [ ]:
# reshaping ESCS results for visualization
plot_escs = (
    math_by_escs
    .rename(columns={"OECD": "OECD Comparison Group"})
    .rename_axis("ESCS Range")
    .reset_index()
    .melt(
        id_vars="ESCS Range",
        var_name="Analysis Group",
        value_name="Math Score"
    )
)

# keeping ESCS ranges in their correct order
plot_escs["ESCS Range"] = pd.Categorical(
    plot_escs["ESCS Range"],
    categories=escs_labels,
    ordered=True
)


# plotting mathematics performance across ESCS ranges
plt.figure(figsize=(10, 5))

sns.lineplot(
    data=plot_escs,
    x="ESCS Range",
    y="Math Score",
    hue="Analysis Group",
    style="Analysis Group",
    markers=True,
    dashes=False,
    markersize=8,
    linewidth=2.5,
    palette="colorblind"
)

plt.title(
    "Figure 3: Mathematics Performance Across Common ESCS Ranges",
    fontweight="bold"
)
plt.xlabel("Socioeconomic Index (ESCS)")
plt.ylabel("Weighted Mean Mathematics Score")
plt.xticks(rotation=15)
plt.grid(linestyle=":", alpha=0.5)
plt.legend(title="Analysis Group")

plt.tight_layout()
plt.show()

## 9D. Gender and Material Shortage Comparisons

In [ ]:
# preparing gender results for visualization
gender_plot_data = (
    subgroup_math_summary[
        ["Female Math", "Male Math"]
    ]
    .rename(index={"OECD": "OECD Comparison Group"})
    .reset_index()
    .melt(
        id_vars="Analysis Group",
        var_name="Gender",
        value_name="Math Score"
    )
)

gender_plot_data["Gender"] = (
    gender_plot_data["Gender"]
    .str.replace(" Math", "", regex=False)
)


# preparing material shortage results for visualization
material_shortage_plot_data = (
    math_by_material_shortage
    .rename(columns={"OECD": "OECD Comparison Group"})
    .rename_axis("Shortage Level")
    .reset_index()
    .melt(
        id_vars="Shortage Level",
        var_name="Analysis Group",
        value_name="Math Score"
    )
)

material_shortage_plot_data["Shortage Level"] = pd.Categorical(
    material_shortage_plot_data["Shortage Level"],
    categories=["Low", "Moderate", "High"],
    ordered=True
)


# plotting gender and educational material shortage results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Figure 4A: mathematics performance by gender
sns.barplot(
    data=gender_plot_data,
    x="Analysis Group",
    y="Math Score",
    hue="Gender",
    ax=axes[0],
    palette="colorblind"
)

axes[0].set_title(
    "Figure 4A: Mathematics Performance by Gender",
    fontweight="bold"
)
axes[0].set_xlabel("")
axes[0].set_ylabel("Weighted Mean Mathematics Score")
axes[0].set_ylim(0, 550)
axes[0].grid(axis="y", linestyle=":", alpha=0.5)


# Figure 4B: mathematics performance by material shortage
sns.barplot(
    data=material_shortage_plot_data,
    x="Shortage Level",
    y="Math Score",
    hue="Analysis Group",
    ax=axes[1],
    palette="colorblind"
)

axes[1].set_title(
    "Figure 4B: Mathematics Performance by Material Shortage",
    fontweight="bold"
)
axes[1].set_xlabel("Educational Material Shortage")
axes[1].set_ylabel("Weighted Mean Mathematics Score")
axes[1].set_ylim(0, 550)
axes[1].grid(axis="y", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

# 10. Export Cleaned Data and Results for Repository

In [ ]:
# exporting analysis-ready cleaned data and result tables
from pathlib import Path

cleaned_data_dir = Path("cleaned_data")
results_dir = Path("results")

cleaned_data_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

# keeping only the groups and variables used in the final analysis
analysis_columns = [
    "analysis_group",
    "country",
    "school_id",
    "student_id",
    "socioeconomic_index",
    "is_male",
    "grade_repetition",
    "student_weight",
    *math_cols,
    *read_cols,
    *science_cols,
    "school_location",
    "edu_material_shortage",
    "staff_shortage"
]

student_analysis_clean = (
    student_school_data[
        student_school_data["analysis_group"].isin(groups)
    ][analysis_columns]
    .copy()
)

school_analysis_clean = (
    school_clean[
        school_clean["analysis_group"].isin(groups)
    ][
        [
            "analysis_group",
            "country",
            "school_id",
            "school_location",
            "edu_material_shortage",
            "staff_shortage"
        ]
    ]
    .copy()
)

# parquet keeps the cleaned datasets much smaller than CSV
student_analysis_clean.to_parquet(
    cleaned_data_dir / "student_analysis_clean.parquet",
    index=False
)

school_analysis_clean.to_parquet(
    cleaned_data_dir / "school_analysis_clean.parquet",
    index=False
)

# exporting the main result tables
performance_summary.to_csv(
    results_dir / "performance_summary.csv"
)

covariate_summary.to_csv(
    results_dir / "covariate_summary.csv"
)

subgroup_math_summary.to_csv(
    results_dir / "subgroup_math_summary.csv"
)

math_by_escs.to_csv(
    results_dir / "math_by_escs.csv"
)

math_by_material_shortage.to_csv(
    results_dir / "math_by_material_shortage.csv"
)

math_by_school_location.to_csv(
    results_dir / "math_by_school_location.csv"
)

correlation_summary.to_csv(
    results_dir / "correlation_summary.csv"
)

print("Export complete.")
print(f"Student cleaned data: {cleaned_data_dir / 'student_analysis_clean.parquet'}")
print(f"School cleaned data: {cleaned_data_dir / 'school_analysis_clean.parquet'}")
print(f"Result tables: {results_dir}")
